# 10 个问题的谜题：用 Implies 解开互相引用的问答

## 这一章讲什么

这道谜题一共有 10 个问题，麻烦之处在于它们互相引用对方的答案。由于每一题的答案都会出现在别的题目里，因此不能一题一题地顺序求解，只能把十道题一起摆在约束里。

- 每题的回答变成一个 1～5 的整数变量
- “如果答案是某某，那么这条条件成立”，用 `Implies()` 写出来
- 硬穷举要 5¹⁰ ≈ 976 万种，写成约束就不必再数了

整章的重心落在 `Implies()`(蕴含)这一个函数上，下面先看题目的难点在什么地方。

## 问题

```
1. 第一个答案是“b”的问题是哪个？
（a）2；（b）3；（c）4；（d）5；（e）6

2. 唯一一组连续两题答案相同的是？
（a）2，3；（b）3，4；（c）4，5；（d）5，6；（e）6，7；

3. 与本题答案相同的问题是哪个？
（a）1；（b）2；（c）4；（d）7；（e）6

4. 答案是“a”的问题有几个？
（a）0；（b）1；（c）2；（d）3；（e）4

5. 与本题答案相同的问题是哪个？
（a）10；（b）9；（c）8；（d）7；（e）6

6. 答案是“a”的问题个数，与答案是哪个选项的问题个数相同？
（a）b；（b）c；（c）d；（d）e；（e）以上都不是

7. 按字母顺序，本题答案与下一题答案相差几个字母？
（a）4；（b）3；（c）2；（d）1；（e）0。（※a 与 b 相差 1 个字母）

8. 答案是元音字母（a, e）的问题有几个？
（a）2；（b）3；（c）4；（d）5；（e）6。（※元音字母是 a 和 e）

9. 答案是辅音字母（b, c, d）的问题个数是？
（a）质数；（b）阶乘数；（c）平方数；（d）立方数；（e）5 的倍数

10. 本题的答案是？
（a）a；（b）b；（c）c；（d）d；（e）e
```

## 难点在哪里

- 条件彼此咬着：Q3 的答案得跟另一题的答案对上；Q6 更绕，比较的是“答 a 的题数”和“答其他选项的题数”。
- 组合数量很大：1 题 5 个选项，排到 10 题，老老实实全探索是 $5^{10} = 9{,}765{,}625$ 种。
- 有的条件不像等式：Q7 要的是答案之差的绝对值，Q9 要的是答辅音的题数落在质数、平方数、阶乘数、5 的倍数这几个之中。

这样的量级人手是数不过来的，因此只能把条件照原样交给 Z3，让求解器去搜。

## 思路

问题 $i$ 的答案用一个整数变量 `q[i]` 装着，取值为 1～5，对应关系是 1=a、2=b、3=c、4=d、5=e。

十个问题的句式完全一样，即如果 Q_i 的答案是某某，那么这一条条件成立，翻译过来正是 `Implies()`。下面拿 Q4 举例，写成代码就是下面的样子。

```python
solver.add([Implies(q[4] == ans, count_a() == value) for ans, value in zip(choices, [0, 1, 2, 3, 4])])
```

这一行里装了五条约束：Q4 答 a(1) 时 a 的个数是 0，答 b(2) 时是 1，一路排到答 e(5) 时是 4，正好把五个选项都覆盖到了。

Q10 有一点特别，“本题的答案是哪一个”这一问挑哪个选项都不矛盾，因为它只指向自己，因此我们一条约束也不给它。

In [1]:
from z3 import *

# 选项 a〜e 用整数 1〜5 表示（不是 Z3 变量，只是编号）
a, b, c, d, e = range(1, 6)
choices = [a, b, c, d, e]

# 10 个问题的答案
q = {i: Int(f'q{i}') for i in range(1, 11)}
solver = Solver()

# 每个答案都在 1〜5（a〜e 之一）
for x in q.values():
    solver.add(1 <= x, x <= 5)


def first_b(idx):
    """“第一个答案是 b 的是问题 idx”这个条件"""
    return And([q[i] != b for i in range(1, idx)] + [q[idx] == b])


def only_equals(idx):
    """“相邻且答案相同的只有 idx 和 idx+1 这一组”这个条件"""
    return And([q[i] == q[i + 1] if i == idx else q[i] != q[i + 1]
                for i in range(1, 10)])


def count_answer(ans):
    """答案为 ans 的问题个数"""
    return Sum([x == ans for x in q.values()])


def is_in(expr, values):
    """expr 等于 values 中的某一个"""
    return Or([expr == v for v in values])


# Q1: 第一个答案是 b 的问题
solver.add([Implies(q[1] == ans, first_b(idx)) for ans, idx in zip(choices, [2, 3, 4, 5, 6])])

# Q2: 唯一一组相邻且答案相同
solver.add([Implies(q[2] == ans, only_equals(idx)) for ans, idx in zip(choices, [2, 3, 4, 5, 6])])

# Q3: 与本题答案相同的问题
solver.add([Implies(q[3] == ans, q[3] == q[idx]) for ans, idx in zip(choices, [1, 2, 4, 7, 6])])

# Q4: a 的个数
solver.add([Implies(q[4] == ans, count_answer(a) == value) for ans, value in zip(choices, [0, 1, 2, 3, 4])])

# Q5: 与本题答案相同的问题
solver.add([Implies(q[5] == ans, q[5] == q[idx]) for ans, idx in zip(choices, [10, 9, 8, 7, 6])])

# Q6: a 的个数与哪个答案的个数相同
solver.add(
    Implies(q[6] == a, count_answer(a) == count_answer(b)),
    Implies(q[6] == b, count_answer(a) == count_answer(c)),
    Implies(q[6] == c, count_answer(a) == count_answer(d)),
    Implies(q[6] == d, count_answer(a) == count_answer(e)),
    Implies(q[6] == e, And(count_answer(a) != count_answer(b),
                           count_answer(a) != count_answer(c),
                           count_answer(a) != count_answer(d),
                           count_answer(a) != count_answer(e))),
)

# Q7: 本题与下一题的答案，按字母顺序相差几个字母
solver.add([Implies(q[7] == ans, Abs(q[7] - q[8]) == diff) for ans, diff in zip(choices, [4, 3, 2, 1, 0])])

# Q8: 元音（a, e）的个数
solver.add([Implies(q[8] == ans, count_answer(a) + count_answer(e) == value)
            for ans, value in zip(choices, [2, 3, 4, 5, 6])])

# Q9: 辅音（b, c, d）的个数是质数、阶乘数、平方数、立方数、5 的倍数之一
consonants = count_answer(b) + count_answer(c) + count_answer(d)
solver.add([Implies(q[9] == ans, is_in(consonants, values)) for ans, values in
            zip(choices, [[2, 3, 5, 7],      # 质数
                          [1, 2, 6],         # 阶乘数 (1!, 2!, 3!)
                          [1, 4, 9],         # 平方数
                          [1, 8],            # 立方数
                          [5, 10]])])        # 5 的倍数

# Q10 是自指的，所以不加约束

if solver.check() == sat:
    m = solver.model()
    answers = [m[q[i]].as_long() for i in range(1, 11)]
    print("答案:", "".join("abcde"[v - 1] for v in answers))
    for i, v in enumerate(answers, 1):
        print(f"Q{i} = {'abcde'[v - 1]}")
else:
    print("无解")

答案: cdebeedcba
Q1 = c
Q2 = d
Q3 = e
Q4 = b
Q5 = e
Q6 = e
Q7 = d
Q8 = c
Q9 = b
Q10 = a


跑出来的答案只有 `cdebeedcba` 这一种。

| Q1 | Q2 | Q3 | Q4 | Q5 | Q6 | Q7 | Q8 | Q9 | Q10 |
|---|---|---|---|---|---|---|---|---|---|
| c | d | e | b | e | e | d | c | b | a |

注意，Q9 所说的“5 的倍数”，实际取的是 5 和 10，1 不算 5 的倍数，因此没有放进去。

想确认真的只有一个解，办法是这样的：把刚找到的答案当作禁止条件加回去，再调用一次 `check()`，此时返回 `unsat` 就说明别的答案一个也不存在。

## 什么是 Implies

`Implies(P, Q)` 表示“P 真则 Q 真”，在 Z3 里它并没有引入新的约束，与下面的式子等价。

$$
\text{Implies}(P, Q) \equiv \neg P \vee Q
$$

| $P$ | $Q$ | $\text{Implies}(P, Q)$ |
|:---:|:---:|:---:|
| False | False | True |
| False | True | True |
| True | False | False |
| True | True | True |

- `P` 假的时候，`Q` 说什么都不影响结果，整体仍然为真(这是逻辑的规矩，假能推出任何结论)。
- 只有当 `P` 真、`Q` 假时，整体才变成假。

正因为这一条性质，我们才能写出“只在某个条件成立时，另一条条件才生效”这种约束。

下面拿一个小例子验证一下它的行为。其中 `x` 和 `y` 都取 1～4，然后加上两条约束试试。

- `Implies(x >= 3, y <= 2)` … x 落到 3 或 4 时，y 就被限制到 1 或 2
- `Implies(x <= 2, y >= 3)` … 反过来，x 落到 1 或 2 时，y 只能取 3 或 4

| `x` | `y` 能取的值 |
|---|---|
| 1, 2 | 3, 4 |
| 3, 4 | 1, 2 |

数一下组合，一共有 $2 \times 2 \times 2 = 8$ 种。

In [2]:
x, y = Ints('x y')
s = Solver()
s.add(1 <= x, x <= 4)
s.add(1 <= y, y <= 4)
s.add(Implies(x >= 3, y <= 2))
s.add(Implies(x <= 2, y >= 3))

while s.check() == sat:
    m = s.model()
    print(m)
    s.add(Or([d() != m[d] for d in m]))   # 禁止已找到的解，继续找下一个解

[y = 1, x = 3]
[y = 2, x = 4]
[x = 2, y = 3]
[x = 2, y = 4]
[x = 1, y = 3]
[y = 2, x = 3]
[x = 4, y = 1]
[x = 1, y = 4]


8 种解一种不少地都出来了，`Implies` 在这里起的是条件约束的作用，看得很清楚。

顺便记一条经验：像这道题这样每个选项挂一条条件的题目，把 `Implies` 按选项个数摆成一排就是了。想写得紧凑一些，例如用 `zip(choices, [...])`，一行就能把选项和条件配好对，不必逐条手写。

## 小结

- 自指谜题看着古怪，只要把每题的答案变成变量，剩下的就是普通的约束问题
- “如果答案是某某，则……”这类句子交给 `Implies()`，有几个选项就摆几条
- 计数用 `Sum([x == ans for x in q.values()])`、绝对值用 `Abs`、“属于其中某一个”用 `Or`，都能当约束使用
- 976 万种的全探索，写成约束之后 Z3 一瞬间就解完了